### Notebook 9: OpenTelemetry Tracing + SQLite Logging

ADK emits OpenTelemetry spans for every agent run, tool call, and model request. This notebook captures those spans alongside Python log records and persists both to a local SQLite database for offline analysis.

Two tables in one database:
- `traces` -- OpenTelemetry spans (agent name, tool calls, durations, model info)
- `logs` -- Python log records (timestamp, level, message, logger name)

Writes use synchronous sqlite3 (background threads). Reads use aiosqlite for async querying in notebook cells.

### 1. Imports and Configuration

In [3]:
import sys
import os
import time
import json
import sqlite3
import logging
from pathlib import Path
from datetime import datetime

from IPython.display import display, Markdown
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / "environment" / ".env")

DB_PATH = str(PROJECT_ROOT / "database" / "telemetry.db")
MODEL   = "gemini-3-flash-preview"

### 2. SQLite Database Setup

In [4]:
# -----------------------------------------------------------------------
# Create the SQLite database with two tables.
# This runs once and subsequent runs append to the same database.
# -----------------------------------------------------------------------

conn = sqlite3.connect(DB_PATH)
conn.execute("""
    CREATE TABLE IF NOT EXISTS traces (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        trace_id        TEXT,
        span_id         TEXT,
        parent_span_id  TEXT,
        name            TEXT,
        kind            TEXT,
        start_time      TEXT,
        end_time        TEXT,
        duration_ms     REAL,
        status          TEXT,
        attributes      TEXT,
        events          TEXT,
        resource        TEXT,
        recorded_at     TEXT DEFAULT (datetime('now'))
    )
""")
conn.execute("""
    CREATE TABLE IF NOT EXISTS logs (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp   TEXT,
        level       TEXT,
        logger      TEXT,
        message     TEXT,
        trace_id    TEXT,
        span_id     TEXT,
        recorded_at TEXT DEFAULT (datetime('now'))
    )
""")
conn.commit()
conn.close()

log = logging.getLogger(__name__)
log.info("Database ready at %s", DB_PATH)

### 3. Custom OpenTelemetry Span Exporter

A custom `SpanExporter` that writes OTel spans to the `traces` table. ADK emits spans for agent runs, LLM calls, and tool executions. The `BatchSpanProcessor` calls `export()` in a background thread, so synchronous sqlite3 is safe here.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import (
    BatchSpanProcessor,
    SpanExporter,
    SpanExportResult,
)
from opentelemetry.sdk.resources import Resource


class SQLiteSpanExporter(SpanExporter):
    """Exports OpenTelemetry spans to a SQLite database."""

    def __init__(self, db_path: str):
        self.db_path = db_path

    def export(self, spans):
        """Write a batch of spans to the traces table."""
        conn = sqlite3.connect(self.db_path)
        for span in spans:
            # Calculate duration in milliseconds
            duration_ms = (span.end_time - span.start_time) / 1_000_000

            # Serialize attributes and events to JSON
            attrs = dict(span.attributes) if span.attributes else {}
            events = [
                {
                    "name": e.name,
                    "timestamp": str(e.timestamp),
                    "attributes": dict(e.attributes) if e.attributes else {},
                }
                for e in span.events
            ] if span.events else []

            resource = dict(span.resource.attributes) if span.resource else {}

            conn.execute(
                """INSERT INTO traces
                   (trace_id, span_id, parent_span_id, name, kind,
                    start_time, end_time, duration_ms, status,
                    attributes, events, resource)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (
                    format(span.context.trace_id, "032x"),
                    format(span.context.span_id, "016x"),
                    format(span.parent.span_id, "016x") if span.parent else None,
                    span.name,
                    str(span.kind),
                    str(span.start_time),
                    str(span.end_time),
                    duration_ms,
                    span.status.status_code.name if span.status else "UNSET",
                    json.dumps(attrs, default=str),
                    json.dumps(events, default=str),
                    json.dumps(resource, default=str),
                ),
            )
        conn.commit()
        conn.close()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        pass


# -----------------------------------------------------------------------
# Configure OpenTelemetry with our SQLite exporter.
# This MUST happen before any ADK imports that use tracing.
# -----------------------------------------------------------------------

resource = Resource.create({"service.name": "adk-exploration"})
provider = TracerProvider(resource=resource)
provider.add_span_processor(
    BatchSpanProcessor(SQLiteSpanExporter(DB_PATH))
)
trace.set_tracer_provider(provider)

log.info("OpenTelemetry configured with SQLite exporter")

### 4. Custom Log Handler

A logging handler that writes Python log records to the `logs` table.
Captures the active OTel trace/span IDs so log entries can be correlated
with trace spans.

In [6]:
class SQLiteLogHandler(logging.Handler):
    """Writes log records to a SQLite database."""

    def __init__(self, db_path: str):
        super().__init__()
        self.db_path = db_path

    def emit(self, record):
        """Write one log record to the logs table."""
        try:
            # Get current trace context if available
            current_span = trace.get_current_span()
            ctx = current_span.get_span_context() if current_span else None
            trace_id = format(ctx.trace_id, "032x") if ctx and ctx.trace_id else None
            span_id = format(ctx.span_id, "016x") if ctx and ctx.span_id else None

            conn = sqlite3.connect(self.db_path)
            conn.execute(
                """INSERT INTO logs
                   (timestamp, level, logger, message, trace_id, span_id)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                (
                    datetime.fromtimestamp(record.created).isoformat(),
                    record.levelname,
                    record.name,
                    self.format(record),
                    trace_id,
                    span_id,
                ),
            )
            conn.commit()
            conn.close()
        except Exception:
            self.handleError(record)


# -----------------------------------------------------------------------
# Attach the SQLite handler to the root logger so all log output
# (from ADK, LiteLLM, our code, etc.) is captured.
# -----------------------------------------------------------------------

sqlite_handler = SQLiteLogHandler(DB_PATH)
sqlite_handler.setFormatter(
    logging.Formatter("%(asctime)s  %(levelname)-8s  %(name)s  %(message)s")
)
logging.getLogger().addHandler(sqlite_handler)

# Also configure console output
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt = "%H:%M:%S",
)
logging.getLogger("google.genai").setLevel(logging.ERROR)

log.info("SQLite log handler attached")

### 5. Run Agent

In [8]:
from google.adk.agents   import Agent
from google.adk.sessions import InMemorySessionService
from google.adk.runners  import Runner
from google.genai         import types

from config import APP_NAME, USER_ID, strip_emojis
from tools  import ALL_TOOLS

In [9]:
# -----------------------------------------------------------------------
# Simple agent run to generate telemetry data.
# -----------------------------------------------------------------------

agent = Agent(
    model       = MODEL,
    name        = "telemetry_test_agent",
    description = "Financial advisor for telemetry testing.",
    instruction = (
        "You are a financial advisor. Use the available tools to answer "
        "the user's question. Be concise. No emojis."
    ),
    tools = ALL_TOOLS,
)

runner = Runner(
    agent           = agent,
    app_name        = APP_NAME,
    session_service = InMemorySessionService(),
)

session = await runner.session_service.create_session(
    app_name = APP_NAME,
    user_id  = USER_ID,
)

content = types.Content(
    role  = "user",
    parts = [types.Part(text=(
        "500k house, 100k down, 6.0% rate, 30yr. "
        "Buyer: 120k income, 600/mo debts, 710 credit. "
        "Calculate the payment and assess the risk."
    ))],
)

final_text = None

async for event in runner.run_async(
    user_id     = USER_ID,
    session_id  = session.id,
    new_message = content,
):
    author = getattr(event, "author", "unknown")
    if event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, "function_call") and part.function_call:
                log.info("[%s] tool_call: %s", author, part.function_call.name)
            if hasattr(part, "function_response") and part.function_response:
                log.info("[%s] tool_response: %s", author, part.function_response.name)
    if event.is_final_response() and event.content and event.content.parts:
        for part in event.content.parts:
            if hasattr(part, "text") and part.text and not getattr(part, "thought", False):
                final_text = strip_emojis(part.text)

if final_text:
    display(Markdown(f"**{author}:**\n\n{final_text}"))

# Flush the span processor to ensure all spans are exported
provider.force_flush()
log.info("Telemetry flushed.")

**telemetry_test_agent:**

The monthly mortgage payment is $2,398.20.

**Risk Assessment Summary:**
*   **Recommendation:** Approve
*   **Risk Level:** Low
*   **Key Metrics:**
    *   **DTI Ratio:** 30.0% (Good)
    *   **LTV Ratio:** 80% (Good - No PMI required)
    *   **Credit Score:** 710 (Good)
    *   **Payment-to-Income:** 24.0% (Good)

The loan application is strong due to a solid 20% down payment and a debt-to-income ratio that remains well within the standard limits.

### 6. Query Traces

Read back the captured OpenTelemetry spans from SQLite using aiosqlite.
Each row represents one span -- an agent run, a tool call, or an LLM request.

In [10]:
import aiosqlite

async def query_db(sql, params=()):
    async with aiosqlite.connect(DB_PATH) as db:
        db.row_factory = aiosqlite.Row
        async with db.execute(sql, params) as cursor:
            rows = await cursor.fetchall()
            return [dict(row) for row in rows]

# -----------------------------------------------------------------------
# All trace spans from this session
# -----------------------------------------------------------------------

traces = await query_db(
    "SELECT name, kind, duration_ms, status, attributes FROM traces ORDER BY id"
)

if traces:
    df = pd.DataFrame(traces)
    df["duration_ms"] = df["duration_ms"].round(1)
    # Parse key attributes for display
    df["attrs_preview"] = df["attributes"].apply(
        lambda a: str(json.loads(a))[:100] if a else ""
    )
    display(df[["name", "kind", "duration_ms", "status", "attrs_preview"]])
else:
    log.info("no traces found")

,name,kind,duration_ms,status,attrs_preview
0,execute_tool calculate_monthly_payment,SpanKind.INTERNAL,0.2,UNSET,"{'gen_ai.operation.name': 'execute_tool', 'gen..."
1,generate_content gemini-3-flash-preview,SpanKind.INTERNAL,1657.9,UNSET,"{'gen_ai.system': 'gemini', 'gen_ai.operation...."
2,call_llm,SpanKind.INTERNAL,1658.1,UNSET,"{'gen_ai.system': 'gcp.vertex.agent', 'gen_ai...."
3,execute_tool assess_loan_risk,SpanKind.INTERNAL,0.2,UNSET,"{'gen_ai.operation.name': 'execute_tool', 'gen..."
4,generate_content gemini-3-flash-preview,SpanKind.INTERNAL,1557.0,UNSET,"{'gen_ai.system': 'gemini', 'gen_ai.operation...."
5,call_llm,SpanKind.INTERNAL,1557.1,UNSET,"{'gen_ai.system': 'gcp.vertex.agent', 'gen_ai...."
6,generate_content gemini-3-flash-preview,SpanKind.INTERNAL,1792.8,UNSET,"{'gen_ai.system': 'gemini', 'gen_ai.operation...."
7,call_llm,SpanKind.INTERNAL,1792.9,UNSET,"{'gen_ai.system': 'gcp.vertex.agent', 'gen_ai...."
8,invoke_agent telemetry_test_agent,SpanKind.INTERNAL,5016.1,UNSET,"{'gen_ai.operation.name': 'invoke_agent', 'gen..."
9,invocation,SpanKind.INTERNAL,5016.2,UNSET,{}


### 7. Query Logs

In [11]:
# -----------------------------------------------------------------------
# All log records captured during the agent run
# -----------------------------------------------------------------------

logs = await query_db(
    "SELECT timestamp, level, logger, message, trace_id FROM logs ORDER BY id"
)

if logs:
    df_logs = pd.DataFrame(logs)
    df_logs["message"] = df_logs["message"].str[:120]
    display(df_logs[["timestamp", "level", "logger", "message"]])
else:
    log.info("no logs found")

,timestamp,level,logger,message
0,2026-04-22T07:25:05.654970,WARNING,google_genai.types,"2026-04-22 07:25:05,654 WARNING google_gena..."


### 8. Trace-Log Correlation

Because the log handler captures the active OTel trace ID, logs and spans can be joined. This shows which log entries occurred within which trace span which is useful for debugging agent behaviour.

In [12]:
# -----------------------------------------------------------------------
# Join logs with traces on trace_id to show correlation
# -----------------------------------------------------------------------

correlated = await query_db("""
    SELECT
        l.timestamp,
        l.level,
        l.message,
        t.name AS span_name,
        t.duration_ms
    FROM logs l
    LEFT JOIN traces t ON l.trace_id = t.trace_id
    WHERE l.trace_id IS NOT NULL
    ORDER BY l.id
    LIMIT 30
""")

if correlated:
    df_corr = pd.DataFrame(correlated)
    df_corr["message"] = df_corr["message"].str[:100]
    display(df_corr)
else:
    log.info("no correlated entries found")

,timestamp,level,message,span_name,duration_ms
0,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",call_llm,1557.068
1,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",call_llm,1658.131
2,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",call_llm,1792.871
3,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",execute_tool assess_loan_risk,0.191
4,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",execute_tool calculate_monthly_payment,0.188
5,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",generate_content gemini-3-flash-preview,1556.980
6,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",generate_content gemini-3-flash-preview,1657.908
7,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",generate_content gemini-3-flash-preview,1792.789
8,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",invocation,5016.232
9,2026-04-22T07:25:05.654970,WARNING,"2026-04-22 07:25:05,654 WARNING google_gena...",invoke_agent telemetry_test_agent,5016.072


### 9. Summary Statistics

In [13]:
# -----------------------------------------------------------------------
# Aggregate statistics from the telemetry database
# -----------------------------------------------------------------------

stats = await query_db("""
    SELECT
        name,
        COUNT(*) as call_count,
        ROUND(AVG(duration_ms), 1) as avg_duration_ms,
        ROUND(MIN(duration_ms), 1) as min_duration_ms,
        ROUND(MAX(duration_ms), 1) as max_duration_ms
    FROM traces
    GROUP BY name
    ORDER BY avg_duration_ms DESC
""")

if stats:
    display(pd.DataFrame(stats))

log_stats = await query_db("""
    SELECT level, COUNT(*) as count
    FROM logs
    GROUP BY level
    ORDER BY count DESC
""")

if log_stats:
    display(pd.DataFrame(log_stats))

,name,call_count,avg_duration_ms,min_duration_ms,max_duration_ms
0,invocation,2,5234.4,5016.2,5452.7
1,invoke_agent telemetry_test_agent,2,5234.1,5016.1,5452.2
2,call_llm,6,1742.1,1557.1,2259.9
3,generate_content gemini-3-flash-preview,6,1742.0,1557.0,2259.8
4,execute_tool assess_loan_risk,2,0.3,0.2,0.4
5,execute_tool calculate_monthly_payment,2,0.2,0.2,0.2


,level,count
0,WARNING,1


### 10. Cleanup

In [14]:
# -----------------------------------------------------------------------
# Shut down the tracer provider to flush any remaining spans.
# The database file persists at PROJECT_ROOT/telemetry.db for
# later analysis.
# -----------------------------------------------------------------------

provider.shutdown()
log.info("Tracer provider shut down.")
log.info("Telemetry database: %s", DB_PATH)